<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-05/AI_Modul_5.4_Praktikum_Features_dan_Labels.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 5.4: Praktikum Komputasi - Features dan Labels dalam Machine Learning
Program Studi Sarjana Sains Data / Kecerdasan Buatan / Teknik Pertanian  
Fakultas Teknologi Pertanian, Institut Pertanian STIPER Yogyakarta

---

### Tujuan Praktikum:
1. Memahami representasi komputasi matriks desain fitur $X \in \mathbb{R}^{n \times d}$ dan vektor label target $y$.
2. Memproses berbagai tipologi fitur (numerik kontinu, diskrit, kategorikal nominal, dan ordinal).
3. Mendemonstrasikan secara empiris fenomena *Curse of Dimensionality* dan konsentrasi jarak Euclidean.
4. Menerapkan dan membandingkan tiga metode seleksi fitur: *Filter*, *Wrapper* (RFE), dan *Embedded* (L1 Regularization & Tree Importance).
5. Menganalisis bahaya fatal *target leakage* (*kebocoran target*) dan mengimplementasikan protokol pencegahan dengan Scikit-Learn `Pipeline`.

In [ ]:
import os
os.environ['MPLBACKEND'] = 'Agg'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from scipy.spatial.distance import pdist

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 150
print("Environment komputasi berhasil diinisialisasi tanpa error.")


## 1. Representasi Matriks Desain Fitur ($X$) dan Vektor Target ($y$)

Dalam komputasi numerik, data agro-biosains direpresentasikan sebagai pasangan matriks dua dimensi $X$ dan vektor target satu dimensi $y$.
Mari kita simulasikan dataset agronomi kelapa sawit yang mencakup berbagai skala pengukuran variabel.


In [ ]:
np.random.seed(42)
n_samples = 300

# 1. Fitur Numerik Kontinu
curah_hujan = np.random.uniform(1500, 3200, n_samples)
kadar_air_tanah = np.random.uniform(18.0, 42.0, n_samples)
ndvi_kanopi = np.random.uniform(0.45, 0.92, n_samples)

# 2. Fitur Numerik Diskrit
frekuensi_pemupukan = np.random.poisson(lam=4, size=n_samples)
populasi_hama_ulat = np.random.poisson(lam=2, size=n_samples)

# 3. Fitur Kategorikal Ordinal
kelas_lereng_kategori = ['Datar', 'Landai', 'Agak Curam', 'Curam']
kelas_lereng = np.random.choice(kelas_lereng_kategori, size=n_samples, p=[0.4, 0.35, 0.15, 0.1])

# 4. Fitur Kategorikal Nominal
varietas_bibit_kategori = ['Dumpy', 'Yangambi', 'Avros']
varietas_bibit = np.random.choice(varietas_bibit_kategori, size=n_samples, p=[0.35, 0.40, 0.25])

# 5. Variabel Target: Produktivitas Tandan Buah Segar (TBS) - Ton/Ha/Tahun
produktivitas_tbs = (
    0.005 * curah_hujan +
    0.35 * kadar_air_tanah +
    14.0 * ndvi_kanopi +
    0.8 * frekuensi_pemupukan -
    1.2 * populasi_hama_ulat +
    np.random.normal(0, 1.2, n_samples)
)

# 6. Target Klasifikasi Biner: Status Produktivitas Tinggi (>= median)
status_produktivitas_tinggi = (produktivitas_tbs >= np.median(produktivitas_tbs)).astype(int)

df_agro = pd.DataFrame({
    'Curah_Hujan_mm': curah_hujan,
    'Kadar_Air_Tanah_pct': kadar_air_tanah,
    'NDVI_Kanopi': ndvi_kanopi,
    'Frekuensi_Pemupukan': frekuensi_pemupukan,
    'Populasi_Hama': populasi_hama_ulat,
    'Kelas_Lereng': kelas_lereng,
    'Varietas_Bibit': varietas_bibit,
    'Produktivitas_TBS_TonHa': produktivitas_tbs,
    'Target_Klasifikasi': status_produktivitas_tinggi
})

print("Dimensi Data Agro Mentah:", df_agro.shape)
print(df_agro.head(3))


## 2. Transformasi dan Rekayasa Fitur Heterogen

Variabel kategorikal harus di-encode sebelum diproses algoritma matematika:
- **Ordinal Encoding**: Mempertahankan urutan hierarki topografi (`Datar` < `Landai` < `Agak Curam` < `Curam`).
- **One-Hot Encoding**: Mengonversi variabel nominal (`Dumpy`, `Yangambi`, `Avros`) menjadi representasi biner ortogonal.


In [ ]:
fitur_numerik = ['Curah_Hujan_mm', 'Kadar_Air_Tanah_pct', 'NDVI_Kanopi', 'Frekuensi_Pemupukan', 'Populasi_Hama']
fitur_ordinal = ['Kelas_Lereng']
fitur_nominal = ['Varietas_Bibit']

hierarki_lereng = [['Datar', 'Landai', 'Agak Curam', 'Curam']]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), fitur_numerik),
        ('ord', OrdinalEncoder(categories=hierarki_lereng), fitur_ordinal),
        ('nom', OneHotEncoder(drop='first', sparse_output=False), fitur_nominal)
    ]
)

X_processed = preprocessor.fit_transform(df_agro)
y_klasifikasi = df_agro['Target_Klasifikasi'].values

encoded_nom_cols = preprocessor.named_transformers_['nom'].get_feature_names_out(fitur_nominal).tolist()
nama_fitur_lengkap = fitur_numerik + fitur_ordinal + encoded_nom_cols

print(f"Matriks Desain X Processed: Dimensi = {X_processed.shape} (n = {X_processed.shape[0]}, d = {X_processed.shape[1]})")
print(f"Vektor Label y: Dimensi = {y_klasifikasi.shape}")
print(f"Daftar Kolom Fitur Akhir: {nama_fitur_lengkap}")


## 3. Simulasi Matematis: Curse of Dimensionality & Konsentrasi Jarak Euclidean

Mari kita buktikan secara empiris fenomena konsentrasi jarak Euclidean: seiring bertambahnya dimensi $d$, rasio kontras relatif jarak antar titik mendekati nol:
$$\text{Relative Contrast} = \frac{\max \|\mathbf{x}_i - \mathbf{x}_j\|_2 - \min \|\mathbf{x}_i - \mathbf{x}_j\|_2}{\min \|\mathbf{x}_i - \mathbf{x}_j\|_2} \to 0$$


In [ ]:
dimensi_uji = [2, 5, 10, 20, 50, 100, 200, 500, 1000]
rasio_kontras = []
mean_jarak = []

n_sim = 200
for d in dimensi_uji:
    titik_ruang = np.random.uniform(0, 1, size=(n_sim, d))
    jarak_pasangan = pdist(titik_ruang, metric='euclidean')
    
    d_min = np.min(jarak_pasangan)
    d_max = np.max(jarak_pasangan)
    
    kontras = (d_max - d_min) / d_min
    rasio_kontras.append(kontras)
    mean_jarak.append(np.mean(jarak_pasangan))

fig, ax1 = plt.subplots(figsize=(8, 4))
color = '#1f77b4'
ax1.set_xlabel('Dimensi Ruang Fitur (d)', fontweight='bold')
ax1.set_ylabel('Kontras Relatif Jarak ((d_max - d_min) / d_min)', color=color, fontweight='bold')
ax1.plot(dimensi_uji, rasio_kontras, marker='o', linewidth=2, color=color)
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_xscale('log')

ax2 = ax1.twinx()  
color = '#d62728'
ax2.set_ylabel('Rata-rata Jarak Euclidean Antar-Titik', color=color, fontweight='bold')
ax2.plot(dimensi_uji, mean_jarak, marker='s', linewidth=2, linestyle='--', color=color)
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Simulasi Empiris Curse of Dimensionality & Konsentrasi Jarak Euclidean', fontweight='bold', fontsize=11)
fig.tight_layout()
plt.savefig('docs/assets/simulasi_curse_of_dimensionality_notebook.png', bbox_inches='tight')
plt.close()
print(f"Simulasi selesai. Kontras Relatif d=2: {rasio_kontras[0]:.4f} -> d=1000: {rasio_kontras[-1]:.4f}")


## 4. Metodologi Seleksi Fitur (*Feature Selection*)

Kita membandingkan tiga paradigma:
1. **Filter Method**: ANOVA F-Value (SelectKBest).
2. **Wrapper Method**: Recursive Feature Elimination (RFE) berbasis Logistic Regression.
3. **Embedded Method**: Random Forest Feature Importance.


In [ ]:
X_syn, y_syn = make_classification(
    n_samples=600,
    n_features=50,
    n_informative=8,
    n_redundant=4,
    n_repeated=0,
    n_classes=2,
    random_state=42
)

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_syn, y_syn, test_size=0.3, random_state=42, stratify=y_syn
)

# 1. Baseline: Seluruh 50 Fitur
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train_s)
X_te_sc = scaler.transform(X_test_s)

rf_all = RandomForestClassifier(n_estimators=100, random_state=42)
rf_all.fit(X_tr_sc, y_train_s)
acc_all = accuracy_score(y_test_s, rf_all.predict(X_te_sc))

# 2. Filter Method: SelectKBest (ANOVA F-test, k=10)
selector_filter = SelectKBest(score_func=f_classif, k=10)
X_tr_filter = selector_filter.fit_transform(X_tr_sc, y_train_s)
X_te_filter = selector_filter.transform(X_te_sc)

rf_filter = RandomForestClassifier(n_estimators=100, random_state=42)
rf_filter.fit(X_tr_filter, y_train_s)
acc_filter = accuracy_score(y_test_s, rf_filter.predict(X_te_filter))

# 3. Wrapper Method: RFE (n_features_to_select=10)
rfe = RFE(estimator=LogisticRegression(max_iter=1000, random_state=42), n_features_to_select=10)
X_tr_rfe = rfe.fit_transform(X_tr_sc, y_train_s)
X_te_rfe = rfe.transform(X_te_sc)

rf_rfe = RandomForestClassifier(n_estimators=100, random_state=42)
rf_rfe.fit(X_tr_rfe, y_train_s)
acc_rfe = accuracy_score(y_test_s, rf_rfe.predict(X_te_rfe))

# 4. Embedded Method: Random Forest Importance Top-10
importances = rf_all.feature_importances_
top_idx_rf = np.argsort(importances)[::-1][:10]

rf_embedded = RandomForestClassifier(n_estimators=100, random_state=42)
rf_embedded.fit(X_tr_sc[:, top_idx_rf], y_train_s)
acc_embedded = accuracy_score(y_test_s, rf_embedded.predict(X_te_sc[:, top_idx_rf]))

hasil_seleksi = pd.DataFrame({
    'Metode Seleksi': ['Tanpa Seleksi (50 Fitur)', 'Filter (ANOVA F-Test, 10 Fitur)', 'Wrapper (RFE, 10 Fitur)', 'Embedded (RF Importance, 10 Fitur)'],
    'Akurasi Data Uji': [acc_all, acc_filter, acc_rfe, acc_embedded],
    'Reduksi Dimensi (%)': [0.0, 80.0, 80.0, 80.0]
})

print(hasil_seleksi.to_string(index=False))


## 5. Eksperimen Fatalitas Target Leakage (*Kebocoran Target*)

Target leakage terjadi apabila variabel masukan terkontaminasi oleh informasi masa depan atau turunan langsung dari label target.
Mari kita buktikan mengapa akurasi tinggi pada data latih/uji yang bocor adalah bencana saat diterapkan di lapangan.


In [ ]:
df_leak = pd.DataFrame(X_syn[:, :8], columns=[f'Fitur_{i}' for i in range(8)])
df_leak['Target_Hasil_Panen'] = y_syn

# Kebocoran: Memasukkan variabel pascapanen yang belum ada saat inferensi lapangan!
df_leak['Fitur_Bocor_Pascapanen'] = y_syn * 2.5 + np.random.normal(0, 0.05, len(y_syn))

X_bocor = df_leak.drop(columns=['Target_Hasil_Panen'])
y_target = df_leak['Target_Hasil_Panen']

X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_bocor, y_target, test_size=0.3, random_state=42)

model_bocor = LogisticRegression()
model_bocor.fit(X_tr_l, y_tr_l)
acc_bocor_test = accuracy_score(y_te_l, model_bocor.predict(X_te_l))

# Pada saat operasional nyata di lapangan, fitur bocor tersebut tidak ada (diisi acak/noise)
X_lapangan_rusak = X_te_l.copy()
X_lapangan_rusak['Fitur_Bocor_Pascapanen'] = np.random.normal(1.25, 0.5, len(y_te_l))
acc_operasional_lapangan = accuracy_score(y_te_l, model_bocor.predict(X_lapangan_rusak))

print(f"Akurasi Pengujian di Lab (Mengandung Kebocoran Target) : {acc_bocor_test * 100:.2f}% (Tampak Sempurna)")
print(f"Akurasi Operasional Nyata di Lapangan (Deployment)        : {acc_operasional_lapangan * 100:.2f}% (Gagal Total)")


## 6. Latihan Terpandu Mandiri (Tugas Mahasiswa)

1. **Uji Efek Multikolinearitas**: Hitung matriks korelasi Pearson untuk seluruh fitur numerik pada `df_agro`. Identifikasi pasangan fitur yang memiliki korelasi absolut tinggi ($|r| > 0.60$).
2. **Seleksi Fitur Regresi**: Gunakan dataset regresi `Produktivitas_TBS_TonHa` dan terapkan regularisasi $\ell_1$ (Lasso). Cari nilai hiperparameter $\alpha$ optimal menggunakan validasi silang sehingga memangkas minimal 2 fitur dari matriks desain!
3. **Penyusunan Pipeline Bebas Kebocoran**: Rancang `sklearn.pipeline.Pipeline` yang menggabungkan imputasi nilai hilang, standarisasi, seleksi fitur ANOVA, dan estimator klasifikasi. Jelaskan mengapa penempatan seleksi fitur di dalam `Pipeline` secara otomatis melindungi data uji dari *pre-processing leakage*.
